In [4]:
#default_exp intersect_annotations

# welcome to intersect_annotations

> These functions use bedtools intersect function (run using pybedtools) which can find where annotations of file A match up w/ annotation B



In [1]:
#hide
from nbdev.showdoc import *

In [3]:
#export
import pybedtools
import pandas as pd
import matplotlib as plt
import seaborn as sns

> in order to run pybedtools, you need a bed file, which can be generated w/ "make bed file".

In [2]:
#export
def make_bed_file_given_window(df, num_bp, info, no_strand):
    
    start=df.v_start.astype('int')-num_bp
    end=df.v_start.astype('int')+num_bp
    score=[1]*len(df)
    
    strand = df['strand'].str.replace('r','-')
    strand = df['strand'].str.replace('f','+')
    
    df=df.assign(score=score)
    df=df.assign(strand=strand)

    if no_strand==True:
        forBed=pd.concat([df.chromosome, start, end, df[[info, 'score', 'strand']]], axis=1)
        
    else:
        forBed=pd.concat([df.chromosome, start, end, df[info]], axis=1)
    return forBed 




Types of intersect
--
>"perfect"
 
here is the intersect function sometimes you need a full intersect if you intersect two intron databases, for example. 

> "SNPs"

If you just need to intersect SNPs with an annotations

In [5]:
#export
def intersect_bedtools(bed_to_to_intersect, annot_file, overlap):

    
    annot_dir='/Users/hnjacobs/Dropbox (MIT)/GradSchool/Finuance/sQTL_snakemake_wf/intersect/hg38/'

#intersect using pybedtools
    a = bed_to_to_intersect
    b = pybedtools.example_bedtool(annot_dir+annot_file)
#Then do the intersection with the BedTool.intersect() method:

    if overlap=='perfect':
#r paramater is that 100% of intron needs to intersect
        a_and_b = a.intersect(b,  wb=True, wa=True, r=True, f=0.99, e=False)
    elif overlap=='SNPs': 
        a_and_b = a.intersect(b,  wb=True, wa=True, s=True)
    
    return a_and_b.to_dataframe()
    

> "closest" finds the nearest neighbor of file A in annotation B file

In [1]:
#export
def closest_bedtools(bed_to_intersect, annot_file, annot_dir):

    a = bed_to_intersect
    #import file into pybedtools
    if annot_dir=='':
        annot_dir='/Users/hnjacobs/Dropbox (MIT)/GradSchool/Finuance/sQTL_snakemake_wf/intersect/hg38/'
    b = pybedtools.example_bedtool(annot_dir+annot_file)
    
#Then do the closest with the BedTool.closest() method, calculate distances

    a_and_b = a.closest(b, D="a", s=True)

    
    return a_and_b
    
    

In [4]:
#export
def plot_percent_annot_aganist_pip_bins(df, pip_bin_type, annotation):
# top bar -> sum all values(smoker=No and smoker=Yes) to find y position of the bars
    plt.figure(figsize=(7, 7))
    
    sns.set(font_scale=3) 

# set plot style: grey grid in the background:
    sns.set(style="darkgrid")
    
    sum_df=df.groupby(pip_bin_type)[annotation].sum()
    size_df=df.groupby(pip_bin_type)[annotation].size()
    
    
    fraction_of_annot= sum_df.to_frame()/size_df.to_frame()*100
    
    ax=sns.barplot(data=fraction_of_annot, x=fraction_of_annot.index, y=annotation, palette='mako')
    
    ax.set_title(annotation)

    plt.ylabel('% of variants in' + str('annotation'))